Doctor / PA Staff
       │
       ▼
AI Prior Authorization Agent
       │
       ├── Check patient
       │
       ├── Check insurance
       │
       ├── Check whether PA is required
       │
       ├── Find payer requirements
       │
       ├── Check required documents
       │
       ├── Identify missing information
       │
       ├── Create PA request
       │
       └── Track PA status
                │
                ▼
          SQLite Database

# AI Prior Authorization Agent

## Project Overview

Prior authorization is a healthcare process in which a provider may need approval from a patient's insurance company before certain medical services, procedures, or treatments are covered.

This project builds an AI-powered Prior Authorization Agent that helps automate the administrative workflow.

The assistant can:

- Retrieve patient and insurance information
- Check whether prior authorization is required
- Retrieve payer documentation requirements
- Check whether required documents are available
- Identify missing information
- Create a prior authorization request
- Track the status of submitted requests

The project uses an LLM for natural-language interaction, Python for workflow logic, SQLite for data storage, tool calling for actions, and Gradio for the user interface.

> This is an educational project using synthetic patient data. It does not make medical decisions or determine medical necessity.

## Business Problem

Prior authorization can require healthcare staff to manually:

1. Verify patient insurance information
2. Determine whether authorization is required
3. Find payer-specific requirements
4. Collect supporting documentation
5. Submit authorization requests
6. Track the request status
7. Respond to missing-information requests

These repetitive administrative steps can consume significant staff time and may delay the authorization workflow.

An AI agent can help organize this process by retrieving information, checking requirements, identifying missing documents, creating requests, and tracking their status.

## Project Objective

The objective of this project is to build an AI assistant that helps healthcare staff manage the prior authorization workflow.

The system will:

- Store synthetic patient and insurance information
- Store payer-specific prior authorization requirements
- Check whether prior authorization is required
- Check whether required documents are available
- Identify missing documentation
- Create prior authorization requests
- Track authorization status
- Allow users to interact with the workflow using natural language

The AI does not approve or deny medical treatment. It assists with the administrative workflow and leaves authorization decisions to the payer and healthcare professionals.

## Technology Stack

- **Python** - Application and workflow logic
- **OpenAI API** - Natural language understanding and tool calling
- **SQLite** - Patient, payer, document and authorization data
- **Gradio** - Interactive user interface
- **JSON** - Tool arguments and structured data
- **dotenv** - Secure API key management

In [1]:
# Import Libraries
import os
import json
import sqlite3
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print("OpenAI API Key loaded successfully")
else:
    print("OpenAI API Key not found")

MODEL = "gpt-4.1-mini"

openai = OpenAI()

DB = "prior_authorization.db"

OpenAI API Key loaded successfully


### Design Our Database

We'll use synthetic data only

patients
    │
    ├── patient_id
    ├── name
    ├── insurance
    └── member_id

payer_requirements
    │
    ├── payer
    ├── service
    ├── prior_auth_required
    └── required_documents

patient_documents
    │
    ├── patient_id
    └── document_type

prior_authorizations
    │
    ├── authorization_id
    ├── patient_id
    ├── service
    ├── payer
    └── status

In [3]:
# Create Patient Database
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS patients (
            patient_id TEXT PRIMARY KEY,
            name TEXT,
            insurance TEXT,
            member_id TEXT
        )
    """)

    patients = [
        ("PT1001", "John Carter", "HealthPlus", "HP10001"),
        ("PT1002", "Sarah Miller", "CareFirst", "CF20002"),
        ("PT1003", "Michael Brown", "HealthPlus", "HP30003")
    ]

    cursor.executemany("""
        INSERT OR REPLACE INTO patients
        (patient_id, name, insurance, member_id)
        VALUES (?, ?, ?, ?)
    """, patients)

    conn.commit()

### What this code does

This cell creates a simple patient database using synthetic patient information.

Each patient has:

- A patient ID
- A name
- An insurance payer
- An insurance member ID

For example, patient `PT1001` is covered by `HealthPlus`.

In a real healthcare application this information would normally come from an EHR or patient-management system rather than being manually stored in the notebook.

### Create Payer Requirements

Different insurance plans can have different requirements.


In [4]:
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS payer_requirements (
            payer TEXT,
            service TEXT,
            prior_auth_required TEXT,
            required_documents TEXT,
            PRIMARY KEY (payer, service)
        )
    """)

    requirements = [
        (
            "HealthPlus",
            "MRI",
            "Yes",
            "Physician Order, Clinical Notes, Previous Imaging"
        ),
        (
            "HealthPlus",
            "Physical Therapy",
            "Yes",
            "Physician Order, Clinical Notes"
        ),
        (
            "CareFirst",
            "MRI",
            "Yes",
            "Physician Order, Clinical Notes"
        ),
        (
            "CareFirst",
            "X-Ray",
            "No",
            "None"
        )
    ]

    cursor.executemany("""
        INSERT OR REPLACE INTO payer_requirements
        (payer, service, prior_auth_required, required_documents)
        VALUES (?, ?, ?, ?)
    """, requirements)

    conn.commit()

### What this code does

This cell creates a simplified database of insurance payer requirements.

Different insurance companies may have different prior authorization and documentation requirements for different healthcare services.

For example:

- HealthPlus requires prior authorization for an MRI.
- The simulated HealthPlus MRI workflow requires a physician order, clinical notes, and previous imaging.
- CareFirst does not require prior authorization for an X-Ray in our synthetic example.

These rules are fictional and are used only to demonstrate how an automated prior authorization workflow could operate.

In [5]:
# Create Patient Documents
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS patient_documents (
            patient_id TEXT,
            document_type TEXT
        )
    """)

    documents = [
        ("PT1001", "Physician Order"),
        ("PT1001", "Clinical Notes"),
        ("PT1002", "Physician Order"),
        ("PT1002", "Clinical Notes"),
        ("PT1002", "Previous Imaging"),
        ("PT1003", "Physician Order")
    ]

    cursor.executemany("""
        INSERT INTO patient_documents
        (patient_id, document_type)
        VALUES (?, ?)
    """, documents)

    conn.commit()

### What this code does

This cell stores the supporting documents currently available for each patient.

For example, patient `PT1001` currently has:

- Physician Order
- Clinical Notes

The agent will later compare these documents with the payer's requirements to determine whether anything is missing before a prior authorization request is created.

In [6]:
# Create Prior Authorization table
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS prior_authorizations (
            authorization_id TEXT PRIMARY KEY,
            patient_id TEXT,
            service TEXT,
            payer TEXT,
            status TEXT
        )
    """)

    conn.commit()

### What this code does

This table stores prior authorization requests created by our assistant.

Each request contains:

- Authorization ID
- Patient ID
- Healthcare service
- Insurance payer
- Current status

New requests will initially be stored with a status such as `Pending Review`.

In [7]:
# First Tool: Get Patient Information
def get_patient(patient_id):

    print(f"DATABASE TOOL CALLED: Getting patient {patient_id}", flush=True)

    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()

        cursor.execute("""
            SELECT name, insurance, member_id
            FROM patients
            WHERE patient_id = ?
        """, (patient_id.upper(),))

        result = cursor.fetchone()

        if result:
            name, insurance, member_id = result

            return (
                f"Patient {patient_id}: "
                f"{name}, Insurance: {insurance}, "
                f"Member ID: {member_id}"
            )

        return "Patient not found"

### What this code does

This function retrieves a patient's basic information from the database.

When the AI needs information about a patient, it can call this function using the patient ID.

The function returns the patient's name, insurance payer, and member ID.

In [8]:
get_patient("PT1001")

DATABASE TOOL CALLED: Getting patient PT1001


'Patient PT1001: John Carter, Insurance: HealthPlus, Member ID: HP10001'

### Test Result

The function successfully retrieved the patient's insurance information from the SQLite database.

This confirms that our assistant can access patient information before beginning the prior authorization workflow.

### Check Whether Prior Authorization Is Required

In [9]:
def check_prior_auth_requirement(payer, service):

    print(
        f"DATABASE TOOL CALLED: Checking prior authorization for {service}",
        flush=True
    )

    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()

        cursor.execute("""
            SELECT prior_auth_required, required_documents
            FROM payer_requirements
            WHERE LOWER(payer) = LOWER(?)
            AND LOWER(service) = LOWER(?)
        """, (payer, service))

        result = cursor.fetchone()

        if not result:
            return "No prior authorization requirement data available"

        required, documents = result

        if required == "No":
            return (
                f"Prior authorization is not required for "
                f"{service} with {payer}"
            )

        return (
            f"Prior authorization is required for {service} "
            f"with {payer}. Required documents: {documents}"
        )

### What this code does

This function checks whether the patient's insurance payer requires prior authorization for a particular service.

If authorization is required, the function also returns the supporting documents needed for the request.

This represents the coverage-requirement discovery stage of our simplified workflow.

In [10]:
check_prior_auth_requirement("HealthPlus", "MRI")

DATABASE TOOL CALLED: Checking prior authorization for MRI


'Prior authorization is required for MRI with HealthPlus. Required documents: Physician Order, Clinical Notes, Previous Imaging'

In [11]:
check_prior_auth_requirement("CareFirst", "X-Ray")

DATABASE TOOL CALLED: Checking prior authorization for X-Ray


'Prior authorization is not required for X-Ray with CareFirst'

### Test Result

The system can now distinguish between services that require prior authorization and services that do not based on our synthetic payer rules.

This prevents the workflow from creating unnecessary authorization requests.

In [12]:
# Check Supporting Documents
def check_required_documents(patient_id, payer, service):

    print(
        f"DATABASE TOOL CALLED: Checking documents for {patient_id}",
        flush=True
    )

    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()

        cursor.execute("""
            SELECT required_documents
            FROM payer_requirements
            WHERE LOWER(payer) = LOWER(?)
            AND LOWER(service) = LOWER(?)
        """, (payer, service))

        requirement = cursor.fetchone()

        if not requirement:
            return "No documentation requirements found"

        required_documents = requirement[0]

        if required_documents == "None":
            return "No supporting documents are required"

        required_documents = [
            doc.strip()
            for doc in required_documents.split(",")
        ]

        cursor.execute("""
            SELECT document_type
            FROM patient_documents
            WHERE patient_id = ?
        """, (patient_id.upper(),))

        available_documents = [
            row[0] for row in cursor.fetchall()
        ]

        missing_documents = [
            doc for doc in required_documents
            if doc not in available_documents
        ]

        if missing_documents:
            return (
                "Missing documents: "
                + ", ".join(missing_documents)
            )

        return "All required documents are available"

### What this code does

This function compares the payer's required documents with the documents currently available for the patient.

The workflow is:

1. Find the payer's documentation requirements.
2. Retrieve the patient's available documents.
3. Compare the two lists.
4. Identify any missing documents.

If documents are missing, the request should not be submitted yet.

This helps reduce incomplete prior authorization requests.

### Test The Document Check

PT1001 has:
✓ Physician Order
✓ Clinical Notes

HealthPlus MRI requires:
✓ Physician Order
✓ Clinical Notes
✗ Previous Imaging

In [13]:
check_required_documents("PT1001","HealthPlus","MRI")

DATABASE TOOL CALLED: Checking documents for PT1001


'Missing documents: Previous Imaging'

### Test Result

The system correctly identified that patient `PT1001` is missing `Previous Imaging` for the simulated HealthPlus MRI authorization.

Instead of blindly submitting an incomplete request, the agent can now identify what information needs to be collected first.

In [14]:
# Create an Authorization Request
def create_prior_authorization(
    authorization_id,
    patient_id,
    service,
    payer
):

    print(
        f"DATABASE TOOL CALLED: Creating authorization {authorization_id}",
        flush=True
    )

    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()

        cursor.execute("""
            SELECT authorization_id
            FROM prior_authorizations
            WHERE authorization_id = ?
        """, (authorization_id.upper(),))

        if cursor.fetchone():
            return f"Authorization {authorization_id} already exists"

        cursor.execute("""
            INSERT INTO prior_authorizations
            (authorization_id, patient_id, service, payer, status)
            VALUES (?, ?, ?, ?, ?)
        """, (
            authorization_id.upper(),
            patient_id.upper(),
            service,
            payer,
            "Pending Review"
        ))

        conn.commit()

        return (
            f"Authorization {authorization_id} created "
            f"with status Pending Review"
        )

### What this code does

This function creates a new prior authorization request.

Before creating the request, it checks whether the authorization ID already exists.

If it is new, the request is saved in the database with the status:

`Pending Review`

This simulates submitting a prior authorization request into the administrative workflow.

In [15]:
# Retrieve Authorization Status
def get_prior_auth_status(authorization_id):

    print(
        f"DATABASE TOOL CALLED: Getting status for {authorization_id}",
        flush=True
    )

    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()

        cursor.execute("""
            SELECT patient_id, service, payer, status
            FROM prior_authorizations
            WHERE authorization_id = ?
        """, (authorization_id.upper(),))

        result = cursor.fetchone()

        if not result:
            return "Authorization not found"

        patient_id, service, payer, status = result

        return (
            f"Authorization {authorization_id} for patient "
            f"{patient_id}, service {service}, payer {payer}, "
            f"has status {status}"
        )

### What this code does

This function retrieves the current status of a prior authorization request.

Healthcare staff can use the authorization ID to check whether the request is still pending or has moved to another status.

Later, we can expand the project to support additional statuses such as:

- Pending Review
- More Information Required
- Approved
- Denied

In [16]:
get_patient("PT1001")

DATABASE TOOL CALLED: Getting patient PT1001


'Patient PT1001: John Carter, Insurance: HealthPlus, Member ID: HP10001'

In [17]:
check_prior_auth_requirement("HealthPlus","MRI")

DATABASE TOOL CALLED: Checking prior authorization for MRI


'Prior authorization is required for MRI with HealthPlus. Required documents: Physician Order, Clinical Notes, Previous Imaging'

In [18]:
check_required_documents("PT1001","HealthPlus","MRI")

DATABASE TOOL CALLED: Checking documents for PT1001


'Missing documents: Previous Imaging'

In [19]:
create_prior_authorization("PA1001","PT1002","MRI","CareFirst")

DATABASE TOOL CALLED: Creating authorization PA1001


'Authorization PA1001 already exists'

In [20]:
get_prior_auth_status("PA1001")

DATABASE TOOL CALLED: Getting status for PA1001


'Authorization PA1001 for patient PT1002, service MRI, payer CareFirst, has status Pending Review'

User:
"Does John Carter need prior authorization
for an MRI?"

             ↓

            LLM
             ↓
        get_patient()
             ↓
check_prior_auth_requirement()
             ↓
check_required_documents()
             ↓

AI:
"Yes. HealthPlus requires prior authorization
for the MRI. Previous Imaging is currently
missing from the patient's documentation."

## AI Tool Calling

Now that the prior authorization workflow functions are working, we'll connect them to the AI assistant.

This will allow the language model to decide which tool it needs based on the user's request.

In [21]:
# Patient Tool Definition
patient_function = {
    "name": "get_patient",
    "description": "Get a patient's name, insurance payer and member ID using their patient ID.",
    "parameters": {
        "type": "object",
        "properties": {
            "patient_id": {
                "type": "string",
                "description": "The patient ID, for example PT1001"
            }
        },
        "required": ["patient_id"],
        "additionalProperties": False
    }
}

### What this code does

This creates the tool definition for retrieving patient information.

The AI does not directly access the database. Instead, it can request the `get_patient` tool when it needs patient information.

In [22]:
# Prior Authorization Requirement Tool
prior_auth_requirement_function = {
    "name": "check_prior_auth_requirement",
    "description": "Check whether prior authorization is required for a healthcare service with a specific insurance payer.",
    "parameters": {
        "type": "object",
        "properties": {
            "payer": {
                "type": "string",
                "description": "The patient's insurance payer"
            },
            "service": {
                "type": "string",
                "description": "The healthcare service, for example MRI"
            }
        },
        "required": ["payer", "service"],
        "additionalProperties": False
    }
}


### What this code does

This tool allows the AI to check whether a healthcare service requires prior authorization.

The AI provides the insurance payer and healthcare service, and our database returns the prior authorization requirements.

In [23]:
# Document Check Tool
document_check_function = {
    "name": "check_required_documents",
    "description": "Check whether a patient has all documents required for a prior authorization request.",
    "parameters": {
        "type": "object",
        "properties": {
            "patient_id": {
                "type": "string",
                "description": "The patient ID"
            },
            "payer": {
                "type": "string",
                "description": "The patient's insurance payer"
            },
            "service": {
                "type": "string",
                "description": "The healthcare service"
            }
        },
        "required": ["patient_id", "payer", "service"],
        "additionalProperties": False
    }
}

### What this code does

This tool allows the AI to check whether all supporting documents required by the payer are available.

If something is missing, the tool tells the AI which document needs to be collected before submission.

In [24]:
# Create Authorization Tool

create_authorization_function = {
    "name": "create_prior_authorization",
    "description": "Create a new prior authorization request after the requirements and documents have been checked.",
    "parameters": {
        "type": "object",
        "properties": {
            "authorization_id": {
                "type": "string",
                "description": "Authorization ID, for example PA1002"
            },
            "patient_id": {
                "type": "string",
                "description": "The patient ID"
            },
            "service": {
                "type": "string",
                "description": "The healthcare service"
            },
            "payer": {
                "type": "string",
                "description": "The patient's insurance payer"
            }
        },
        "required": [
            "authorization_id",
            "patient_id",
            "service",
            "payer"
        ],
        "additionalProperties": False
    }
}

In [25]:
# Authorization Status Tool
authorization_status_function = {
    "name": "get_prior_auth_status",
    "description": "Get the current status of an existing prior authorization request.",
    "parameters": {
        "type": "object",
        "properties": {
            "authorization_id": {
                "type": "string",
                "description": "The authorization ID, for example PA1001"
            }
        },
        "required": ["authorization_id"],
        "additionalProperties": False
    }
}

### What this code does

This tool allows the AI to retrieve the status of an existing prior authorization request.

A user can simply ask about an authorization ID instead of manually querying the database.

In [26]:
# Giving all tools to AI
tools = [
    {"type": "function", "function": patient_function},
    {"type": "function", "function": prior_auth_requirement_function},
    {"type": "function", "function": document_check_function},
    {"type": "function", "function": create_authorization_function},
    {"type": "function", "function": authorization_status_function}
]

### What this code does

This combines all five functions into one list of tools available to the AI.

The assistant can now choose between:

1. Patient lookup
2. Prior authorization requirement check
3. Document check
4. Prior authorization creation
5. Authorization status lookup

In [27]:
# Handle Tool Calls: - Connect the AI tool names to the actual Python functions
def handle_tool_calls(message):

    responses = []

    for tool_call in message.tool_calls:

        arguments = json.loads(tool_call.function.arguments)
        function_name = tool_call.function.name

        if function_name == "get_patient":
            result = get_patient(
                arguments.get("patient_id")
            )

        elif function_name == "check_prior_auth_requirement":
            result = check_prior_auth_requirement(
                arguments.get("payer"),
                arguments.get("service")
            )

        elif function_name == "check_required_documents":
            result = check_required_documents(
                arguments.get("patient_id"),
                arguments.get("payer"),
                arguments.get("service")
            )

        elif function_name == "create_prior_authorization":
            result = create_prior_authorization(
                arguments.get("authorization_id"),
                arguments.get("patient_id"),
                arguments.get("service"),
                arguments.get("payer")
            )

        elif function_name == "get_prior_auth_status":
            result = get_prior_auth_status(
                arguments.get("authorization_id")
            )

        else:
            result = "Unknown tool"

        responses.append({
            "role": "tool",
            "content": result,
            "tool_call_id": tool_call.id
        })

    return responses

### What this code does

This function connects the AI's tool requests to our Python functions.

When the AI chooses a tool:

1. The tool arguments are read.
2. The correct Python function is called.
3. The database performs the requested operation.
4. The result is returned to the AI.

This is what allows the language model to interact with our healthcare workflow instead of only generating text.

In [28]:
system_message = """
You are a helpful Prior Authorization Assistant for a healthcare provider.

Your job is to help staff manage the administrative prior authorization workflow.

Use your tools whenever patient, payer, document, or authorization information is needed.

When processing a new prior authorization:

1. Retrieve the patient's information.
2. Check whether prior authorization is required for the service.
3. If prior authorization is required, check the required documents.
4. If required documents are missing, explain what is missing and do not create the authorization.
5. Only create the authorization when the required information and documents are available.
6. Never make medical decisions or determine medical necessity.

Keep your answers short, clear and professional.
"""

### What this code does

The system message defines how our AI agent should behave.

It tells the assistant to follow the prior authorization workflow in the correct order and prevents it from creating a request when required documents are missing.

The AI assists with administrative workflow only and does not make medical decisions.

In [29]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]}for h in history]
    messages = ([{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}])

    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools
    )

    while response.choices[0].finish_reason == "tool_calls":
        assistant_message = response.choices[0].message
        responses = handle_tool_calls(assistant_message)
        messages.append(assistant_message)
        messages.extend(responses)

        response = openai.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )

    return response.choices[0].message.content

### What this code does

This is the main AI agent.

When a user sends a message:

1. The question is sent to the language model.
2. The model decides whether it needs a tool.
3. Python executes the requested tool.
4. The result is returned to the model.
5. The model can call another tool if necessary.
6. The process continues until the assistant has enough information to answer.

The `while` loop is important because a prior authorization request may require several tools to complete one workflow.

In [30]:
gr.ChatInterface(fn=chat, type="messages", title="AI Prior Authorization Agent").launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Getting patient pt1001
DATABASE TOOL CALLED: Checking prior authorization for MRI
DATABASE TOOL CALLED: Checking documents for PT1001


PT1001
  ↓
HealthPlus
  ↓
MRI
  ↓
PA Required
  ↓
Check Documents
  ↓
Physician Order ✓
Clinical Notes ✓
Previous Imaging ✗
  ↓
STOP

## Adding Audio Responses

Now we'll add audio so that our Prior Authorization assistant can speak its responses.

In [31]:
# Import Libaries
from pathlib import Path
import tempfile

In [32]:
def talker(message):
    audio_file = "response.mp3"

    with openai.audio.speech.with_streaming_response.create(
        model="tts-1",
        voice="onyx",
        input=message
    ) as response:
        response.stream_to_file(audio_file)

    return audio_file

## Connecting Audio to the AI Assistant

Now we'll connect the text-to-speech function to our Prior Authorization Agent.

After the AI completes the prior authorization workflow, its final response will also be converted into audio.

In [33]:
def chat_with_audio(message, history):
    response = chat(message, history)
    audio_file = talker(response)

    return response, audio_file

### What this code does

This function combines our AI agent with the audio function.

The `chat()` function handles the prior authorization workflow and generates the final answer.

The `talker()` function then converts that answer into speech.

The function returns both the written response and the audio response.

In [34]:
with gr.Blocks() as demo:

    gr.Markdown("# AI Prior Authorization Agent")
    gr.Markdown(
        "An AI assistant for checking prior authorization requirements, "
        "documents, requests and authorization status."
    )

    chatbot = gr.Chatbot(type="messages")

    message = gr.Textbox(
        label="Message",
        placeholder="Ask about a patient or prior authorization..."
    )

    audio = gr.Audio(
        label="Audio Response",
        autoplay=True
    )

    clear = gr.Button("Clear")

    def respond(user_message, history):

        response, audio_file = chat_with_audio(
            user_message,
            history
        )

        history = history + [
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": response}
        ]

        return "", history, audio_file

    message.submit(
        respond,
        inputs=[message, chatbot],
        outputs=[message, chatbot, audio]
    )

    clear.click(
        lambda: ("", [], None),
        outputs=[message, chatbot, audio]
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Getting patient pt1001
DATABASE TOOL CALLED: Checking prior authorization for MRI
DATABASE TOOL CALLED: Checking documents for PT1001
